# Automatic HE Parameter Selection

This notebook demonstrates how to use `hadal_flow` to automatically choose low
level parameters (like the plaintext modulus and ciphertext moduli) for the BGV
HE scheme. While these parameters can be chosen manually, as shown in other
examples, it is convenient to let `hadal_flow` choose them.

Since the HE parameters depend on the depth of a computation, they must be
fixed before the computation starts. `hadal_flow` does this by extending
TensorFlow's graph compiler, grappler,  with some convenient
homomorphic-encryption (HE) specific features, one of which is automatic
parameter selection.

As such, automatic parameter selection is only available when using TensorFlow's
deferred execution mode (graph mode). This way, the graph is available for
inspection (to estimate ciphertext noise growth) and modification (to inject
generated parameters) before it is executed.

In [ ]:
import tensorflow as tf
import hadal_flow

In [2]:
a = [1, 2, 3]
b = [4, 5, 6]

Here we define the function we'd like to compute. TensorFlow will first trace
this function (without executing it) to build a graph of the computation. Then,
during a graph compiler optimization pass, `hadal_flow` will replace the
"autocontext" placeholder Op with parameters generated for this specific
computation based on statistical estimation of the noise growth and the initial
plaintext size.

Note, the `create_autocontext64` function must be called from inside a
`tf.function` in order to execute in deferred mode.

In [ ]:
@tf.function
def foo(cleartext_a, cleartext_b):
    shell_context = hadal_flow.create_autocontext64(
        log2_cleartext_sz=4,  # Maximum size of the cleartexts (including the scaling factor).
        scaling_factor=1,  # The scaling factor (analagous to fixed-point but not necessarily base 2).
        noise_offset_log2=0,  # Extra buffer for noise growth.
    )
    key = hadal_flow.create_key64(shell_context)
    a = hadal_flow.to_encrypted(cleartext_a, key, shell_context)
    b = hadal_flow.to_shell_plaintext(cleartext_b, shell_context)

    intermediate = a * b
    result = hadal_flow.to_tensorflow(intermediate, key)
    return result

In [ ]:
hadal_flow.enable_optimization()  # Enable the autoparameter graph optimization pass.

a = [1, 2, 3]
b = [4, 5, 6]
c = foo(a, b)

print(c)

`hadal_flow` selected a plaintext modulus `t` to be at least 4 bits, and ciphertext
modulus `Q` as a produt of smaller moduli `qs` for representation of ciphertexts
in RNS (residual number system). `Q` is chosen to be large enough to support
noise growth in the computation without overflowing. Since this computation is
small, only one ciphertext modulus is needed.

Note that `hadal_flow` treats the first dimension of data as the packing dimension
of the BGV scheme (the slotting dimension). When the function is first traced,
the size of this dimension is unknown, because the ring degree of the
ciphertexts has not been chosen yet as it depends on `Q`, which depends on the
estimated noise growth. In the example above, the three elements of the input
vectors are packed into this first dimension for efficiency purposes. The
remaining slots in the ciphertexts went unused.